# 04 — Feature Engineering

BQ01: dự đoán `price_vnd` cho BĐS bán tại TP.HCM. Notebook này khóa feature set sau EDA và tài liệu hóa leakage policy.

In [1]:
from pathlib import Path
import sys, pandas as pd
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from dap_project.features.build_features import add_preliminary_model_features, select_model_features, build_feature_selection_report
cleaned = sorted((PROJECT_ROOT/'data/processed').glob('hcm_realestate_cleaned_*.parquet'))[-1]
df = add_preliminary_model_features(pd.read_parquet(cleaned))
print(cleaned)
print(df.shape)

C:\Users\LEGION\Downloads\DAP_Final_Project_BT1_BT2\DAP_Final_Project\data\processed\hcm_realestate_cleaned_20260912_163121.parquet
(6086, 36)


## 1. Availability và feature set

In [2]:
selected = select_model_features(df, min_non_null_rate=0.70)
selected

{'numeric': ['area_m2'], 'categorical': ['property_type', 'legacy_area']}

In [3]:
report = build_feature_selection_report(df, selected)
report[['feature','dtype','missing_rate','cardinality','used_in_model','role','reason']]

,feature,dtype,missing_rate,cardinality,used_in_model,role,reason
0,area_m2,float64,0.0000,1209,True,model_feature,"Đủ availability, có ý nghĩa dự báo và không gâ..."
1,legacy_area,str,0.0000,22,True,model_feature,"Đủ availability, có ý nghĩa dự báo và không gâ..."
2,property_type,str,0.0000,6,True,model_feature,"Đủ availability, có ý nghĩa dự báo và không gâ..."
3,area_size_band,category,0.0000,4,False,excluded,Không thuộc feature set đã khóa cho model v1.
4,area_text,str,0.0000,1529,False,excluded,Không thuộc feature set đã khóa cho model v1.
5,crawl_page,int64,0.0000,350,False,excluded,Không thuộc feature set đã khóa cho model v1.
6,crawled_at,str,0.0000,1,False,excluded,Không thuộc feature set đã khóa cho model v1.
7,current_ward,str,0.0588,102,False,excluded,Không thuộc feature set đã khóa cho model v1.
8,description,float64,1.0000,0,False,excluded,Text có thể chứa trực tiếp giá/giá m² và làm m...
9,is_non_listing_url,bool,0.0000,1,False,excluded,Không thuộc feature set đã khóa cho model v1.


## 2. Leakage policy

Không dùng target hoặc biến suy ra từ target (`price_text`, `price_per_m2`, `price_billion`, `log_price_vnd`), identifier/text (`listing_id`, `url`, `title`, `description`) làm predictor. Preprocessing chỉ fit trên train ở BT4.

## 3. Đa cộng tuyến

Model v1 dùng một biến số chính `area_m2`; `log_area_m2` giữ cho EDA nhưng không đưa đồng thời vào model mặc định để tránh dư thừa mạnh. Categorical được one-hot với drop first trong pipeline.

In [4]:
audit = __import__('json').loads((PROJECT_ROOT/'reports/modeling/modeling_data_audit.json').read_text(encoding='utf-8'))
print('Final numeric:', selected['numeric'])
print('Final categorical:', selected['categorical'])
print('Target:', audit['target'])

Final numeric: ['area_m2']
Final categorical: ['property_type', 'legacy_area']
Target: price_vnd


## 4. Kết luận

Feature set model v1 được đóng băng ở `area_m2`, `property_type`, `legacy_area`. Các feature chi tiết chỉ được thêm khi availability production đủ tốt và vẫn phải qua leakage check.